# Examen 2
> Moctezuma Ramirez Diego Rafael

In [1]:
# Data Wrangling
import numpy as np
import pandas as pd

# Data viz
import plotly.express as px
from sklearn import set_config

# Data preprocessing
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import  StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer

# Model performance
from sklearn.metrics import roc_auc_score

# Modeling
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split, cross_val_score

# Enviroment setup
set_config(display="diagram")
pd.set_option("display.max_columns", 50)
pd.set_option('display.float_format', lambda x: '%.4f' % x)

## Funciones Auxiliares

In [2]:
def frecuencias(df, caracteristicas):
    for caracteristica in caracteristicas:
        print(f"Caracteristica: {caracteristica}")
        abs_ = df[caracteristica].value_counts(dropna=False).to_frame().rename(columns={"count": "Frecuencia absoluta"})
        rel_ = df[caracteristica].value_counts(dropna=False, normalize= True).to_frame().rename(columns={"proportion": "Frecuencia relativa"})
        freq = abs_.join(rel_)
        freq["Frecuencia Acumulada"] = freq["Frecuencia absoluta"].cumsum()
        freq["Acumulada %"] = freq["Frecuencia relativa"].cumsum()
        freq["Frecuencia absoluta"] = freq["Frecuencia absoluta"].map(lambda x: f"{x:,.0f}")
        freq["Frecuencia relativa"] = freq["Frecuencia relativa"].map(lambda x: f"{x:,.2%}")
        freq["Frecuencia Acumulada"] = freq["Frecuencia Acumulada"].map(lambda x: f"{x:,.0f}")
        freq["Acumulada %"] = freq["Acumulada %"].map(lambda x: f"{x:,.2%}")
        display(freq)

## Carga de datos

In [3]:
df = pd.read_csv("Clasificacion/train_default.csv", sep="|")

In [4]:
df.sample(5)

,CUSTOMER_ID,LIMIT_BAL,SEX,EDUCATION,MARRIAGE,AGE,PAY_0,PAY_2,PAY_3,PAY_4,PAY_5,PAY_6,BILL_AMT1,BILL_AMT2,BILL_AMT3,BILL_AMT4,BILL_AMT5,BILL_AMT6,PAY_AMT1,PAY_AMT2,PAY_AMT3,PAY_AMT4,PAY_AMT5,PAY_AMT6,default.payment.next.month
4612,1452,280000.0000,2,1,1,36,-1,0,-1,-1,-1,-1,17951.0000,18915.0000,7926.0000,17965.0000,24432.0000,10805.0000,6094.0000,7966.0000,18059.0000,24554.0000,18860.0000,17313.0000,0
2038,8248,110000.0000,2,1,2,26,0,0,0,0,-2,-2,104804.0000,107979.0000,93600.0000,0.0000,0.0000,0.0000,4988.0000,1872.0000,0.0000,0.0000,0.0000,0.0000,0
39,9769,150000.0000,1,2,1,57,2,2,2,2,2,2,14496.0000,15546.0000,16303.0000,15744.0000,16582.0000,17166.0000,1600.0000,1300.0000,0.0000,1100.0000,1000.0000,1000.0000,1
2700,25324,220000.0000,2,1,2,28,0,0,0,0,0,0,215779.0000,196299.0000,202596.0000,198845.0000,194885.0000,92905.0000,8600.0000,11000.0000,6000.0000,10082.0000,3400.0000,3825.0000,0
3701,22007,100000.0000,2,3,1,35,0,0,0,0,0,0,47843.0000,48885.0000,49904.0000,50774.0000,51957.0000,52991.0000,2111.0000,2130.0000,2000.0000,2000.0000,1898.0000,2000.0000,0


In [5]:
df.dtypes

CUSTOMER_ID                     int64
LIMIT_BAL                     float64
SEX                             int64
EDUCATION                       int64
MARRIAGE                        int64
AGE                             int64
PAY_0                           int64
PAY_2                           int64
PAY_3                           int64
PAY_4                           int64
PAY_5                           int64
PAY_6                           int64
BILL_AMT1                     float64
BILL_AMT2                     float64
BILL_AMT3                     float64
BILL_AMT4                     float64
BILL_AMT5                     float64
BILL_AMT6                     float64
PAY_AMT1                      float64
PAY_AMT2                      float64
PAY_AMT3                      float64
PAY_AMT4                      float64
PAY_AMT5                      float64
PAY_AMT6                      float64
default.payment.next.month      int64
dtype: object

In [6]:
df.isna().sum()

CUSTOMER_ID                   0
LIMIT_BAL                     0
SEX                           0
EDUCATION                     0
MARRIAGE                      0
AGE                           0
PAY_0                         0
PAY_2                         0
PAY_3                         0
PAY_4                         0
PAY_5                         0
PAY_6                         0
BILL_AMT1                     0
BILL_AMT2                     0
BILL_AMT3                     0
BILL_AMT4                     0
BILL_AMT5                     0
BILL_AMT6                     0
PAY_AMT1                      0
PAY_AMT2                      0
PAY_AMT3                      0
PAY_AMT4                      0
PAY_AMT5                      0
PAY_AMT6                      0
default.payment.next.month    0
dtype: int64

## EDA

### Filtrado de variables

In [7]:
ls_discretas = ["SEX","EDUCATION","MARRIAGE","PAY_0","PAY_2","PAY_3","PAY_4","PAY_5","PAY_6"]
ls_continuas = ["BILL_AMT1","BILL_AMT2","BILL_AMT3","BILL_AMT4","BILL_AMT5","BILL_AMT6","PAY_AMT1","PAY_AMT2","PAY_AMT3","PAY_AMT4","PAY_AMT5","PAY_AMT6","LIMIT_BAL","AGE"]
ls_drop = ["CUSTOMER_ID"]

target = "default.payment.next.month"

### EDA Discreto

In [8]:
frecuencias(df, ls_discretas)

Caracteristica: SEX


,Frecuencia absoluta,Frecuencia relativa,Frecuencia Acumulada,Acumulada %
SEX,,,,
2,"3,338",59.34%,"3,338",59.34%
1,"2,287",40.66%,"5,625",100.00%


Caracteristica: EDUCATION


,Frecuencia absoluta,Frecuencia relativa,Frecuencia Acumulada,Acumulada %
EDUCATION,,,,
2,"2,646",47.04%,"2,646",47.04%
1,"1,985",35.29%,"4,631",82.33%
3,908,16.14%,"5,539",98.47%
5,55,0.98%,"5,594",99.45%
4,23,0.41%,"5,617",99.86%
6,5,0.09%,"5,622",99.95%
0,3,0.05%,"5,625",100.00%


Caracteristica: MARRIAGE


...

,Frecuencia absoluta,Frecuencia relativa,Frecuencia Acumulada,Acumulada %
PAY_4,,,,
0,"3,079",54.74%,"3,079",54.74%
-1,"1,067",18.97%,"4,146",73.71%
-2,829,14.74%,"4,975",88.44%
2,579,10.29%,"5,554",98.74%
3,32,0.57%,"5,586",99.31%
7,15,0.27%,"5,601",99.57%
4,11,0.20%,"5,612",99.77%
5,11,0.20%,"5,623",99.96%
1,1,0.02%,"5,624",99.98%


Caracteristica: PAY_5


,Frecuencia absoluta,Frecuencia relativa,Frecuencia Acumulada,Acumulada %
PAY_5,,,,
0,"3,135",55.73%,"3,135",55.73%
-1,"1,075",19.11%,"4,210",74.84%
-2,863,15.34%,"5,073",90.19%
2,488,8.68%,"5,561",98.86%
3,25,0.44%,"5,586",99.31%
4,20,0.36%,"5,606",99.66%
7,16,0.28%,"5,622",99.95%
5,3,0.05%,"5,625",100.00%


Caracteristica: PAY_6


,Frecuencia absoluta,Frecuencia relativa,Frecuencia Acumulada,Acumulada %
PAY_6,,,,
0,"3,037",53.99%,"3,037",53.99%
-1,"1,073",19.08%,"4,110",73.07%
-2,953,16.94%,"5,063",90.01%
2,495,8.80%,"5,558",98.81%
3,39,0.69%,"5,597",99.50%
4,11,0.20%,"5,608",99.70%
7,10,0.18%,"5,618",99.88%
6,7,0.12%,"5,625",100.00%


### EDA Continuo

In [9]:
for caracteristica in ls_continuas:
    fig = px.histogram(df, x=caracteristica, nbins=50, title=f"Histograma de {caracteristica}",template = "plotly_dark")
    fig.show()

## Normalización

> Education

In [10]:
redundantes_e = [0,5,6]
df['EDUCATION'] = df['EDUCATION'].replace(redundantes_e, 4)

In [11]:
df["EDUCATION"].value_counts(True)

EDUCATION
2   0.4704
1   0.3529
3   0.1614
4   0.0153
Name: proportion, dtype: float64

> Marriage

In [12]:
redundantes_m = [0]
df['MARRIAGE'] = df['MARRIAGE'].replace(redundantes_m, 3)

In [13]:
df["MARRIAGE"].value_counts(True)

MARRIAGE
2   0.5328
1   0.4537
3   0.0135
Name: proportion, dtype: float64

## Data Cleaning

In [14]:
shape_out = df.shape

In [15]:
df[ls_continuas].describe(percentiles=[0.01, 0.05, 0.95, 0.99])

,BILL_AMT1,BILL_AMT2,BILL_AMT3,BILL_AMT4,BILL_AMT5,BILL_AMT6,PAY_AMT1,PAY_AMT2,PAY_AMT3,PAY_AMT4,PAY_AMT5,PAY_AMT6,LIMIT_BAL,AGE
count,5625.0000,5625.0000,5625.0000,5625.0000,5625.0000,5625.0000,5625.0000,5625.0000,5625.0000,5625.0000,5625.0000,5625.0000,5625.0000,5625.0000
mean,51448.8933,49271.4887,47006.7145,43191.0172,40070.0018,38397.5936,5841.6871,5926.7707,5136.6795,4808.7209,4739.2476,5655.7310,166694.4000,35.6055
std,74728.9150,71991.2107,69336.9429,64375.2968,60675.8275,59653.0592,20267.1903,22867.1899,18888.2695,16992.3403,15329.0867,20460.8710,130805.0707,9.2084
min,-15308.0000,-67526.0000,-15910.0000,-81334.0000,-36156.0000,-209051.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,10000.0000,21.0000
1%,-156.5600,-203.5200,-156.5600,-201.7600,-222.0000,-351.1200,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,10000.0000,22.0000
5%,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,20000.0000,24.0000
50%,22582.0000,20830.0000,19953.0000,19079.0000,17966.0000,16496.0000,2146.0000,2044.0000,1800.0000,1500.0000,1500.0000,1500.0000,140000.0000,34.0000
95%,200406.2000,194017.2000,186847.6000,177282.8000,164848.0000,162372.6000,17918.6000,19027.4000,17331.0000,15612.2000,15467.8000,18130.6000,450000.0000,53.0000
99%,371226.4000,359551.2000,336984.2400,307209.5200,292164.5200,284765.6000,63165.4800,78269.9600,63676.2000,69818.8000,59438.6800,94370.2800,500000.0000,62.0000
max,746814.0000,605943.0000,577015.0000,565669.0000,524315.0000,496915.0000,873552.0000,1215471.0000,889043.0000,621000.0000,326889.0000,527143.0000,800000.0000,79.0000


In [16]:
dic_outliers = {}
for caracteristica in ls_continuas:
    aux = df[caracteristica].quantile(0.99) + df[caracteristica].std() * 4
    max_val = df[caracteristica].max()
    if max_val > aux:
        dic_outliers[caracteristica] = df[caracteristica].quantile(0.99)

In [17]:
dic_outliers

{'BILL_AMT1': np.float64(371226.40000000136),
 'BILL_AMT4': np.float64(307209.52000000014),
 'PAY_AMT1': np.float64(63165.48000000003),
 'PAY_AMT2': np.float64(78269.96000000037),
 'PAY_AMT3': np.float64(63676.20000000065),
 'PAY_AMT4': np.float64(69818.80000000016),
 'PAY_AMT5': np.float64(59438.68000000005),
 'PAY_AMT6': np.float64(94370.2800000002)}

In [18]:
for caracteristica, perc in dic_outliers.items():
    df = df[(df[caracteristica] <= perc) | (df[caracteristica].isna())]

In [19]:
df.shape[0] / shape_out[0] 

0.9393777777777778

### Variables altamente vacías

In [20]:
df.isna().mean().sort_values()

CUSTOMER_ID                  0.0000
PAY_AMT5                     0.0000
PAY_AMT4                     0.0000
PAY_AMT3                     0.0000
PAY_AMT2                     0.0000
PAY_AMT1                     0.0000
BILL_AMT6                    0.0000
BILL_AMT5                    0.0000
BILL_AMT4                    0.0000
BILL_AMT3                    0.0000
BILL_AMT2                    0.0000
PAY_AMT6                     0.0000
BILL_AMT1                    0.0000
PAY_5                        0.0000
PAY_4                        0.0000
PAY_3                        0.0000
PAY_2                        0.0000
PAY_0                        0.0000
AGE                          0.0000
MARRIAGE                     0.0000
EDUCATION                    0.0000
SEX                          0.0000
LIMIT_BAL                    0.0000
PAY_6                        0.0000
default.payment.next.month   0.0000
dtype: float64

### Variables unarias

In [21]:
df.nunique()

CUSTOMER_ID                   5284
LIMIT_BAL                       64
SEX                              2
EDUCATION                        4
MARRIAGE                         3
AGE                             52
PAY_0                           11
PAY_2                           10
PAY_3                           11
PAY_4                            9
PAY_5                            8
PAY_6                            8
BILL_AMT1                     4581
BILL_AMT2                     4493
BILL_AMT3                     4413
BILL_AMT4                     4328
BILL_AMT5                     4230
BILL_AMT6                     4144
PAY_AMT1                      2246
PAY_AMT2                      2193
PAY_AMT3                      2103
PAY_AMT4                      1937
PAY_AMT5                      1897
PAY_AMT6                      1901
default.payment.next.month       2
dtype: int64

In [22]:
df.std()

CUSTOMER_ID                    8679.1737
LIMIT_BAL                    124263.4919
SEX                               0.4906
EDUCATION                         0.7372
MARRIAGE                          0.5229
AGE                               9.2427
PAY_0                             1.1488
PAY_2                             1.2218
PAY_3                             1.2231
PAY_4                             1.2070
PAY_5                             1.1701
PAY_6                             1.1831
BILL_AMT1                     60774.6353
BILL_AMT2                     58834.2244
BILL_AMT3                     56390.9624
BILL_AMT4                     52200.7284
BILL_AMT5                     49595.9171
BILL_AMT6                     48431.2456
PAY_AMT1                       6392.9397
PAY_AMT2                       6941.7214
PAY_AMT3                       6138.5599
PAY_AMT4                       6362.6846
PAY_AMT5                       5921.7636
PAY_AMT6                       8320.1706
default.payment.

## Modelado

In [23]:
columnas = df.columns.to_list()
columnas.remove(target)
X = df[columnas]
y = df[target]

> Standard escaler

In [24]:
ss = StandardScaler()

> Train-test split

In [25]:
X_train, X_test, y_train, y_test = train_test_split(X, y)

y_train = y_train.values.ravel()
y_test = y_test.values.ravel()

In [26]:
transformer = ColumnTransformer([
    ("scaler", ss, ls_continuas),
    ("dummies", OneHotEncoder(handle_unknown="ignore"), ls_discretas)
])

### Modelo

In [27]:
rl = LogisticRegression(max_iter=1000)

In [28]:
pipe = Pipeline([
    ('preprocessor', transformer),
    ('model', rl)
])

In [29]:
pipe.fit(X_train, y_train)

,steps,"[('preprocessor', ...), ('model', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('scaler', ...), ('dummies', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [30]:
pipe.score(X_test, y_test)

0.7986373959121877

> Cross validation

In [31]:
cv_scores = cross_val_score(pipe, X_train, y_train, cv=10, scoring='roc_auc')

In [32]:
cv_scores.mean(), cv_scores.std()

(np.float64(0.7736634686352944), np.float64(0.026361911929044118))

## Predicción

### Carga de datos

In [33]:
df_prediccion = pd.read_csv("Clasificacion/val_default.csv", sep="|")

In [34]:
df_prediccion.sample(5)

,CUSTOMER_ID,LIMIT_BAL,SEX,EDUCATION,MARRIAGE,AGE,PAY_0,PAY_2,PAY_3,PAY_4,PAY_5,PAY_6,BILL_AMT1,BILL_AMT2,BILL_AMT3,BILL_AMT4,BILL_AMT5,BILL_AMT6,PAY_AMT1,PAY_AMT2,PAY_AMT3,PAY_AMT4,PAY_AMT5,PAY_AMT6
1311,17895,280000.0000,2,2,2,36,0,0,0,0,0,0,183797.0000,168402.0000,161474.0000,152092.0000,145229.0000,148602.0000,6083.0000,5913.0000,5293.0000,5126.0000,5640.0000,5119.0000
1712,17976,140000.0000,2,2,2,28,2,2,2,0,0,2,26450.0000,28823.0000,28171.0000,28740.0000,30454.0000,29930.0000,3007.0000,0.0000,924.0000,2070.0000,0.0000,3000.0000
1809,29765,500000.0000,1,1,1,39,2,0,0,2,2,2,15776.0000,17467.0000,120972.0000,117088.0000,121283.0000,117583.0000,3004.0000,106000.0000,0.0000,8008.0000,78.0000,10549.0000
1301,29680,20000.0000,1,2,2,25,0,0,0,0,0,2,14989.0000,15999.0000,17043.0000,18236.0000,19376.0000,18958.0000,1263.0000,1307.0000,1500.0000,1408.0000,0.0000,674.0000
589,18750,330000.0000,2,1,1,32,0,0,0,0,0,0,155927.0000,141523.0000,134480.0000,126316.0000,125396.0000,128081.0000,5388.0000,5079.0000,4876.0000,4400.0000,4600.0000,5000.0000


### Normalización

> Education

In [35]:
redundantes_e = [0,5,6]
df_prediccion['EDUCATION'] = df_prediccion['EDUCATION'].replace(redundantes_e, 4)

In [36]:
df_prediccion["EDUCATION"].value_counts(True)

EDUCATION
2   0.4640
1   0.3637
3   0.1568
4   0.0155
Name: proportion, dtype: float64

> Marriage

In [37]:
redundantes_m = [0]
df_prediccion['MARRIAGE'] = df_prediccion['MARRIAGE'].replace(redundantes_m, 3)

In [38]:
df_prediccion["MARRIAGE"].value_counts(True)

MARRIAGE
2   0.5349
1   0.4523
3   0.0128
Name: proportion, dtype: float64

### Data Cleaning

In [39]:
shape_out = df_prediccion.shape

In [40]:
df_prediccion[ls_continuas].describe(percentiles=[0.01, 0.05, 0.95, 0.99])

,BILL_AMT1,BILL_AMT2,BILL_AMT3,BILL_AMT4,BILL_AMT5,BILL_AMT6,PAY_AMT1,PAY_AMT2,PAY_AMT3,PAY_AMT4,PAY_AMT5,PAY_AMT6,LIMIT_BAL,AGE
count,1875.0000,1875.0000,1875.0000,1875.0000,1875.0000,1875.0000,1875.0000,1875.0000,1875.0000,1875.0000,1875.0000,1875.0000,1875.0000,1875.0000
mean,48970.4352,47494.0757,45914.4405,42155.7520,39236.5093,37965.7563,6179.1227,6418.2875,4928.4576,4812.2171,5126.2053,5112.4048,173418.6667,35.3637
std,71369.8207,69948.0722,69316.5221,64962.1534,63046.2045,59886.4530,17796.7773,18806.5000,13853.0701,14250.3164,15413.3223,15785.5632,129849.2401,9.2541
min,-2086.0000,-24704.0000,-61506.0000,-9415.0000,-28335.0000,-17149.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,10000.0000,21.0000
1%,-270.4600,-266.3000,-282.3000,-294.8400,-647.8600,-627.5600,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,10000.0000,22.0000
5%,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,20000.0000,23.0000
50%,19833.0000,19147.0000,18634.0000,17322.0000,16048.0000,15112.0000,2187.0000,2178.0000,1900.0000,1500.0000,1500.0000,1420.0000,150000.0000,34.0000
95%,196583.6000,194234.3000,185394.9000,173085.5000,168828.8000,156899.2000,20171.1000,20912.9000,15974.7000,16981.2000,18222.5000,20000.0000,440000.0000,53.0000
99%,322708.8600,313788.5200,316691.2400,305009.8000,291346.9400,288770.5400,78329.0400,93380.7600,56263.8800,64630.0000,80000.0000,75446.2400,500000.0000,58.0000
max,653062.0000,671563.0000,689643.0000,706864.0000,823540.0000,501370.0000,261524.0000,388126.0000,310852.0000,280695.0000,287982.0000,210000.0000,780000.0000,73.0000


In [41]:
dic_outliers = {}
for caracteristica in ls_continuas:
    aux = df_prediccion[caracteristica].quantile(0.99) + df_prediccion[caracteristica].std() * 4
    max_val = df_prediccion[caracteristica].max()
    if max_val > aux:
        dic_outliers[caracteristica] = df_prediccion[caracteristica].quantile(0.99)

In [42]:
dic_outliers

{'BILL_AMT1': np.float64(322708.86),
 'BILL_AMT2': np.float64(313788.51999999996),
 'BILL_AMT3': np.float64(316691.23999999993),
 'BILL_AMT4': np.float64(305009.8),
 'BILL_AMT5': np.float64(291346.93999999994),
 'PAY_AMT1': np.float64(78329.03999999988),
 'PAY_AMT2': np.float64(93380.75999999995),
 'PAY_AMT3': np.float64(56263.879999999976),
 'PAY_AMT4': np.float64(64630.0),
 'PAY_AMT5': np.float64(80000.0),
 'PAY_AMT6': np.float64(75446.23999999996)}

In [43]:
for caracteristica, perc in dic_outliers.items():
    df_prediccion = df_prediccion[(df_prediccion[caracteristica] <= perc) | (df_prediccion[caracteristica].isna())]

In [44]:
df_prediccion.shape[0] / shape_out[0] 

0.9338666666666666

### Predicción

In [45]:
pipe.fit(X, y)

,steps,"[('preprocessor', ...), ('model', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('scaler', ...), ('dummies', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [46]:
ids = df_prediccion["CUSTOMER_ID"]
df_prediccion.drop(columns=ls_drop, inplace=True)

In [47]:
pred = pipe.predict(df_prediccion)

In [48]:
df_resultado = pd.DataFrame({
    "CARDHOLDER_ID": ids,
    "y_hat": pred
})

In [49]:
df_resultado.sample(5)

,CARDHOLDER_ID,y_hat
1622,25403,1
1395,13680,0
1818,18032,0
1718,4307,0
753,4873,0


In [50]:
df_resultado.to_csv("Respuestas/MoctezumaRamirezDiegoRafael_DefaultPayment.csv", index=False)